# 8. N8N ORCHESTRATION PREPARATION
## Daily Customer Churn Predictor · VivaMarket Brasil

---

**INPUT:** `../data/processed/churn_predictions_YYYYMMDD.parquet`, `../data/processed/churn_explainability_YYYYMMDD.parquet`, and `../models/churn_scoring_package_YYYYMMDD.joblib`

**OUTPUT:** `../data/processed/retention_actions_YYYYMMDD.parquet`, `../n8n/daily_churn_retention_workflow_YYYYMMDD.json`, and `../reports/n8n_orchestration_YYYYMMDD.html`

*A production-minded retention-action payload and an n8n workflow blueprint aligned with the project retention strategy.*


---
## 8.1. STARTING SITUATION


The project now has risk scores, explainability outputs, and a deployment-ready scoring package. The remaining operational step is to define exactly what the daily orchestration should send downstream: who should be contacted, through which channels, with what incentive, and under which guardrails.

This notebook translates the confirmed retention strategy into an executable payload structure aligned with the canonical V2C policy and the professional action catalog defined for VivaMarket Brasil.

---
## 8.2. NOTEBOOK OBJECTIVE


- **Business objective:** convert scored customers into daily retention actions that match the High / Medium / Low framework from the confirmed retention strategy document.
- **Technical objective:** build a workflow-ready payload and a concrete n8n JSON blueprint that can later be implemented with minimal ambiguity.

In [1]:
import json
import logging
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s', force=True)
logger = logging.getLogger('nb08_n8n_orchestration')
logger.info('NB08 started: n8n orchestration preparation.')


2026-05-06 08:32:32,921 | INFO | NB08 started: n8n orchestration preparation.


In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORTS_DIR = PROJECT_ROOT / 'reports'
N8N_DIR = PROJECT_ROOT / 'n8n'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
N8N_DIR.mkdir(parents=True, exist_ok=True)

run_date_tag = datetime.now(ZoneInfo('Europe/Paris')).strftime('%Y%m%d')
prediction_path = sorted(PROCESSED_DIR.glob('churn_predictions_*.parquet'))[-1]
explainability_path = sorted(PROCESSED_DIR.glob('churn_explainability_*.parquet'))[-1]
actions_path = PROCESSED_DIR / f'retention_actions_{run_date_tag}.parquet'
workflow_json_path = N8N_DIR / f'daily_churn_retention_workflow_{run_date_tag}.json'
orchestration_html_path = REPORTS_DIR / f'n8n_orchestration_{run_date_tag}.html'


In [3]:
prediction_df = pd.read_parquet(prediction_path)
explainability_df = pd.read_parquet(explainability_path)
action_df = prediction_df.merge(
    explainability_df[[
        'customer_unique_id', 'snapshot_key', 'top_driver_group', 'recommended_offer_type',
        'recommended_discount_pct', 'free_shipping_flag', 'vip_human_touch_flag', 'ltv_segment'
    ]],
    on=['customer_unique_id', 'snapshot_key'],
    how='left',
    validate='one_to_one',
)

action_df['top_driver_group'] = action_df['top_driver_group'].fillna('unassigned_sample_gap')
action_df['ltv_segment'] = action_df['ltv_segment'].astype(object).fillna('UNASSIGNED')

def normalize_offer(row):
    driver = row['top_driver_group']
    risk = row['risk_tier']
    if risk == 'HIGH':
        if driver == 'recency':
            return 'reactivacion_fuerte'
        if driver == 'frequency':
            return 'compra_recurrente'
        if driver == 'logistics':
            return 'desculpas_prioridade'
        if driver == 'monetary':
            return 'bundle_upsell_exclusivo'
        if driver == 'category':
            return 'cupon_categoria'
        return 'personalizado_categoria'
    if risk == 'MEDIUM':
        return 'nurturing_recomendaciones'
    return 'loyalty_fidelizacion'

action_df['recommended_offer_type'] = action_df.apply(normalize_offer, axis=1)
action_df['recommended_discount_pct'] = action_df['recommended_discount_pct'].fillna(
    action_df['risk_tier'].map({'HIGH': 25, 'MEDIUM': 12, 'LOW': 0})
)
action_df.loc[action_df['risk_tier'].eq('HIGH') & action_df['vip_human_touch_flag'].fillna(False), 'recommended_discount_pct'] = 30
action_df['free_shipping_flag'] = action_df['free_shipping_flag'].fillna(
    action_df['risk_tier'].map({'HIGH': True, 'MEDIUM': True, 'LOW': False})
)
action_df['vip_human_touch_flag'] = action_df['vip_human_touch_flag'].fillna(False)

action_df['primary_channels'] = action_df['risk_tier'].map({
    'HIGH': 'email,push,sms',
    'MEDIUM': 'email,push',
    'LOW': 'email,in_app',
})
action_df['contact_policy'] = action_df['risk_tier'].map({
    'HIGH': 'day0_email_push__day3_sms_if_no_open__day7_escalated_email__day14_feedback_survey',
    'MEDIUM': 'every_3_to_7_days_nurturing',
    'LOW': 'weekly_or_monthly_loyalty_content',
})
action_df['message_focus'] = action_df.apply(
    lambda row: {
        ('HIGH', 'recency'): 'sentimos_sua_falta',
        ('HIGH', 'frequency'): 'volte_com_frequencia',
        ('HIGH', 'logistics'): 'envio_prioritario',
        ('HIGH', 'monetary'): 'kit_exclusivo_valor',
        ('HIGH', 'category'): 'volte_para_categoria',
        ('MEDIUM', 'unassigned_sample_gap'): 'recomendacoes_personalizadas',
    }.get((row['risk_tier'], row['top_driver_group']), 'fidelizacao_relacionamento'),
    axis=1,
)
action_df['control_group_flag'] = False
high_risk_idx = action_df[action_df['risk_tier'] == 'HIGH'].sample(frac=0.15, random_state=42).index
action_df.loc[high_risk_idx, 'control_group_flag'] = True
action_df['send_action_flag'] = ~action_df['control_group_flag']
action_df['offer_code_stub'] = action_df.apply(lambda row: f"{row['risk_tier'][:1]}-{row['snapshot_key']}-{row.name}", axis=1)
action_df['journey_stage_count'] = action_df['risk_tier'].map({'HIGH': 4, 'MEDIUM': 2, 'LOW': 1})
action_df.to_parquet(actions_path, index=False)
logger.info('Retention actions parquet saved to %s', actions_path)
action_df.head()

2026-05-06 08:32:33,038 | INFO | Retention actions parquet saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/retention_actions_20260506.parquet


,customer_unique_id,snapshot_key,snapshot_date,recency_days,total_orders,total_payment_value,orders_30d,orders_90d,observed_target,churn_probability,...,free_shipping_flag,vip_human_touch_flag,ltv_segment,primary_channels,contact_policy,message_focus,control_group_flag,send_action_flag,offer_code_stub,journey_stage_count
0,004288347e5e88a27ded2bb23747066c,20180401,2018-04-01,77,2,354.37,0.0,1.0,1,0.910062,...,False,False,GROWTH,"email,in_app",weekly_or_monthly_loyalty_content,fidelizacao_relacionamento,False,True,L-20180401-0,1
1,00cc12a6d8b578b8ebd21ea4e2ae8b27,20180401,2018-04-01,376,2,126.20,0.0,0.0,1,0.954483,...,False,False,ENTRY,"email,in_app",weekly_or_monthly_loyalty_content,fidelizacao_relacionamento,False,True,L-20180401-1,1
2,011b4adcd54683b480c4d841250a987f,20180401,2018-04-01,45,2,236.30,0.0,1.0,1,0.856885,...,False,False,GROWTH,"email,in_app",weekly_or_monthly_loyalty_content,fidelizacao_relacionamento,False,True,L-20180401-2,1
3,013f4353d26bb05dc6652f1269458d8d,20180401,2018-04-01,124,2,356.39,0.0,0.0,1,0.646774,...,False,False,GROWTH,"email,in_app",weekly_or_monthly_loyalty_content,fidelizacao_relacionamento,False,True,L-20180401-3,1
4,015557c9912277312b9073947804a7ba,20180401,2018-04-01,335,2,315.12,0.0,0.0,1,0.967295,...,True,False,GROWTH,"email,push",every_3_to_7_days_nurturing,fidelizacao_relacionamento,False,True,M-20180401-4,2


In [4]:
workflow_definition = {
    'name': 'Daily Churn Retention Actions',
    'schedule': '0 2 * * * America/Sao_Paulo',
    'description': 'Daily orchestration blueprint for VivaMarket Brasil churn retention actions.',
    'nodes': [
        {'id': 1, 'name': 'Cron Trigger', 'type': 'n8n-nodes-base.cron'},
        {'id': 2, 'name': 'Load Scoring Package', 'type': 'n8n-nodes-base.code'},
        {'id': 3, 'name': 'Read Daily Snapshot', 'type': 'n8n-nodes-base.postgres'},
        {'id': 4, 'name': 'Score Customers', 'type': 'n8n-nodes-base.code'},
        {'id': 5, 'name': 'Apply Retention Rules', 'type': 'n8n-nodes-base.code'},
        {'id': 6, 'name': 'Split Risk Tier', 'type': 'n8n-nodes-base.switch'},
        {'id': 7, 'name': 'Generate Coupon', 'type': 'n8n-nodes-base.httpRequest'},
        {'id': 8, 'name': 'Send Email', 'type': 'n8n-nodes-base.sendGrid'},
        {'id': 9, 'name': 'Send Push', 'type': 'n8n-nodes-base.oneSignal'},
        {'id': 10, 'name': 'Send SMS', 'type': 'n8n-nodes-base.twilio'},
        {'id': 11, 'name': 'Wait for Open Window', 'type': 'n8n-nodes-base.wait'},
        {'id': 12, 'name': 'Log Actions', 'type': 'n8n-nodes-base.postgres'},
        {'id': 13, 'name': 'Error Handler', 'type': 'n8n-nodes-base.emailSend'},
    ],
    'policy': {
        'max_contacts_30d': 4,
        'sms_only_if_email_not_opened': True,
        'control_group_high_risk_share': 0.15,
        'time_windows_local': {'email': '09:00-20:00', 'sms': '10:00-19:00'},
        'high_risk_sequence': ['day0_email_push', 'day3_sms_if_no_open', 'day7_last_chance_email', 'day14_feedback_survey'],
        'medium_risk_sequence': ['recommendation_email', 'followup_push'],
        'low_risk_sequence': ['loyalty_or_value_content'],
    },
}
workflow_json_path.write_text(json.dumps(workflow_definition, indent=2), encoding='utf-8')
logger.info('Workflow JSON saved to %s', workflow_json_path)

2026-05-06 08:32:33,056 | INFO | Workflow JSON saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/n8n/daily_churn_retention_workflow_20260506.json


In [5]:
orchestration_summary = (
    action_df.groupby(['risk_tier', 'recommended_offer_type', 'primary_channels'], observed=False)
    .agg(
        rows_n=('customer_unique_id', 'size'),
        send_action_rows=('send_action_flag', 'sum'),
        control_rows=('control_group_flag', 'sum'),
        avg_discount_pct=('recommended_discount_pct', 'mean'),
    )
    .reset_index()
    .sort_values(['risk_tier', 'rows_n'], ascending=[True, False])
)

html_parts = [
    '<html><head><meta charset="utf-8"><title>N8N Orchestration</title></head><body>',
    '<h1>N8N ORCHESTRATION BLUEPRINT</h1>',
    '<h2>Workflow definition</h2>', f'<pre>{json.dumps(workflow_definition, indent=2)}</pre>',
    '<h2>Action summary</h2>', orchestration_summary.to_html(index=False),
    '<h2>Sample payload</h2>', action_df.head(25).to_html(index=False),
    '</body></html>'
]
orchestration_html_path.write_text('\n'.join(html_parts), encoding='utf-8')
logger.info('Orchestration report saved to %s', orchestration_html_path)
orchestration_summary.head(12)


2026-05-06 08:32:33,078 | INFO | Orchestration report saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/reports/n8n_orchestration_20260506.html


,risk_tier,recommended_offer_type,primary_channels,rows_n,send_action_rows,control_rows,avg_discount_pct
1,HIGH,compra_recurrente,"email,push,sms",530,448,82,25.716981
2,HIGH,reactivacion_fuerte,"email,push,sms",97,84,13,25.206186
0,HIGH,bundle_upsell_exclusivo,"email,push,sms",43,38,5,25.581395
3,LOW,loyalty_fidelizacion,"email,in_app",1673,1673,0,0.000000
4,MEDIUM,nurturing_recomendaciones,"email,push",1003,1003,0,12.000000


---
## 8.3. NOTEBOOK CLOSURE


The orchestration stage now has a concrete action payload, an explicit control-group policy, and a daily n8n workflow blueprint aligned with the confirmed retention strategy. That means the project can move from model outputs to campaign operations without reinterpreting business rules every day.

The final notebook should consolidate these artifacts into a reporting view that helps monitor quality, campaign mix, and future model drift.